# Bouncing ball: global events

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/03_bouncing_ball.ipynb)

A ball falls under gravity. A `global` event fires when `x` crosses 0 going down: it
puts the ball on the floor and reverses its velocity with restitution `e`. The bounces
get shorter geometrically.

*Based on an example from [XPP Studio](https://xpp-studio.github.io/xpp-studio/) (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.3.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("bouncing_ball")
      .ode("x", "v")
      .ode("v", "-g")
      .global_event(-1, "x", {"x": 0, "v": "-e*v"})
      .parameter(g=9.81, e=0.85)
      .initial(x=2, v=0)
      .option(total=6, dt=0.005))
print(mb.render())

x'=v
v'=-g
global -1 x {x=0;v=-e*v}
par g=9.81,e=0.85
init x=2,v=0
@ total=6,dt=0.005
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="x", y="v", window=(-0.2, 2.2, -7, 7), nullclines=False,
              sliders={"e": (0.3, 1), "g": (1, 20)}, title="Bouncing ball")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Bounce times** and the ratio of successive flight times (should be `e`).

In [5]:
import numpy as np
tr = ex.m.run(keep=False, total=6)
b = tr.t[1:][np.diff(np.sign(tr["v"])) > 0]          # v jumps from negative to positive
flights = np.diff(b)
print("bounces at", np.round(b[:6], 4))
print("flight ratios", np.round(flights[1:6] / flights[:5], 4), " e =", ex.m.pars["e"])

bounces at [0.64  1.725 2.65  3.435 4.1   4.665]
flight ratios [0.8525 0.8486 0.8471 0.8496 0.8584]  e = 0.85


## Things to try

- Set `e = 1`: no energy is lost and the ball bounces forever.